# HuatuoGPT-7B — Corrected Inference with v3_simple Prompt
## VLM Medical VQA Benchmark — Notebook 13

**Purpose:** Re-run HuatuoGPT-Vision-7B on the **full SLAKE EN test set (1,061 questions)**
using the **v3_simple prompt** identified as optimal in the Prompt Sensitivity Study (Section 19).

**Why this is needed:**
The main benchmark (Section 11) evaluated all five models using a uniform v2 prompt derived
from the MedGemma Technical Report. That prompt instructs the model to write out reasoning
before stating `Final Answer: X`. HuatuoGPT's Qwen2.5-VL backbone only produces the
`Final Answer:` anchor in 41% of responses — the remaining 59% return a reasoning preamble
as the extracted prediction, with near-zero F1. This is a prompt-template limitation, not an
architectural one.

**v3_simple prompt:** `"Answer concisely in one word or short phrase."` — no anchor required.
On the 200-sample stratified subset this recovered +21.18 pp Token F1 (27.88% → 49.06%).

**Expected outcome:** SLAKE F1 increasing from 47.86% (v2) to approximately 56–62% (v3_simple),
closing approximately half the gap to MedGemma-4B (70.50%).

**Output:** `FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__slake_v3_corrected.jsonl`


In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers==4.51.3 datasets accelerate bitsandbytes sentencepiece pillow
import os, json, re, zipfile, subprocess
from tqdm.auto import tqdm
os.makedirs('/kaggle/working/outputs', exist_ok=True)


In [ ]:
# Cell 2 — Configuration
MODEL_ID      = 'FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL'
MODEL_SAFE    = MODEL_ID.replace('/', '_')
OUTPUT_FILE   = f'/kaggle/working/outputs/{MODEL_SAFE}__slake_v3_corrected.jsonl'
DATASET_NAME  = 'BoKelvin/SLAKE'
SLAKE_IMGS_DIR = '/kaggle/working/slake_imgs'

print(f"Model   : {MODEL_ID}")
print(f"Output  : {OUTPUT_FILE}")


In [ ]:
# Cell 3 — Download SLAKE images (required: not embedded in HF dataset)
import zipfile, subprocess
os.makedirs(SLAKE_IMGS_DIR, exist_ok=True)
zip_path = f'{SLAKE_IMGS_DIR}/imgs.zip'
url = 'https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip'

if not os.path.exists(zip_path):
    print('Downloading SLAKE imgs.zip ...')
    subprocess.run(['curl', '-L', url, '-o', zip_path], check=True)
    print('Download complete.')
else:
    print('Zip already present.')

print('Extracting ...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(SLAKE_IMGS_DIR)

SLAKE_IMG_BASE = os.path.join(SLAKE_IMGS_DIR, 'imgs')
if not os.path.isdir(SLAKE_IMG_BASE):
    SLAKE_IMG_BASE = SLAKE_IMGS_DIR
print(f'Image base: {SLAKE_IMG_BASE}')


In [ ]:
# Cell 4 — Load SLAKE EN test split and pre-load PIL images
from datasets import load_dataset
from PIL import Image

ds = load_dataset(DATASET_NAME, split='test')
en_records = [s for s in ds if s.get('q_lang') == 'en']
print(f'SLAKE EN test: {len(en_records)} records')

def load_slake_image(img_name):
    img_path = os.path.join(SLAKE_IMG_BASE, img_name)
    if not os.path.exists(img_path):
        img_path = os.path.join(SLAKE_IMG_BASE, os.path.basename(img_name))
    return Image.open(img_path).convert('RGB')

# Pre-load images (with error tracking)
samples = []
errors = 0
for i, s in enumerate(en_records):
    try:
        pil_img = load_slake_image(s['img_name'])
        samples.append({
            'orig_idx': i,
            'record': s,
            'image': pil_img,
        })
    except Exception as e:
        errors += 1
        print(f'  Image error idx={i}: {e}')

print(f'Loaded: {len(samples)} ok, {errors} errors')
open_n  = sum(1 for s in samples if s['record']['answer_type'] == 'OPEN')
closed_n = sum(1 for s in samples if s['record']['answer_type'] == 'CLOSED')
print(f'Open: {open_n}, Closed: {closed_n}')


In [ ]:
# Cell 5 — v3_simple prompt builder
# This is the optimal prompt identified in the Prompt Sensitivity Study (Section 19 / T8).
# It avoids the 'Final Answer: X' anchor that HuatuoGPT ignores 59% of the time.

def build_v3_simple(question: str, is_closed: bool) -> str:
    prefix = 'Answer with yes or no only. ' if is_closed else ''
    return f"{prefix}{question} Answer concisely in one word or short phrase."

def extract_answer(raw_output: str) -> str:
    """First non-empty line of the model output — no anchor extraction needed."""
    lines = [l.strip() for l in raw_output.split('\n') if l.strip()]
    return lines[0] if lines else raw_output.strip()

print("Prompt builder ready: v3_simple")
print("Sample (open):", build_v3_simple("What organ is shown?", False))
print("Sample (closed):", build_v3_simple("Is the lung healthy?", True))


In [ ]:
# Cell 6 — Load model (AutoModelForImageTextToText — works for Qwen2.5-VL backbone)
import torch
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)

print(f'Loading {MODEL_ID} ...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map='auto',
    quantization_config=bnb_config,
    trust_remote_code=True,
)
model.eval()
param_b = sum(p.numel() for p in model.parameters()) / 1e9
print(f'Model loaded: {MODEL_ID} ({param_b:.2f}B params)')


In [ ]:
# Cell 7 — Inference loop with resume support
# Runs all 1,061 SLAKE EN test questions with the v3_simple prompt.
# Resume support: if interrupted, re-running will skip already-completed records.

# Load already-completed records
completed = {}
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE) as f:
        for line in f:
            try:
                r = json.loads(line)
                completed[r['idx']] = r
            except: pass
    print(f'Resuming: {len(completed)} / {len(samples)} already done.')

f_out = open(OUTPUT_FILE, 'a')

for entry in tqdm(samples, desc='HuatuoGPT v3_simple'):
    orig_idx = entry['orig_idx']
    if orig_idx in completed:
        continue

    sample  = entry['record']
    pil_img = entry['image']
    is_closed = sample['answer_type'] == 'CLOSED'
    question  = sample['question']
    gt        = str(sample['answer'])

    prompt_text = build_v3_simple(question, is_closed)

    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': pil_img},
            {'type': 'text',  'text': prompt_text},
        ]
    }]

    try:
        text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=text, images=pil_img, return_tensors='pt').to(model.device)
        with torch.inference_mode():
            out_ids = model.generate(**inputs, max_new_tokens=30, do_sample=False)
        in_len  = inputs['input_ids'].shape[-1]
        raw_out = processor.decode(out_ids[0][in_len:], skip_special_tokens=True).strip()
        pred    = extract_answer(raw_out)
    except Exception as e:
        print(f'  Error idx={orig_idx}: {e}')
        raw_out, pred = str(e)[:80], ''

    f_out.write(json.dumps({
        'idx':          orig_idx,
        'question':     question,
        'ground_truth': gt,
        'prediction':   pred,
        'raw_output':   raw_out,
        'is_closed':    is_closed,
        'answer_type':  sample['answer_type'],
        'content_type': sample['content_type'],
        'img_name':     sample['img_name'],
        'model':        MODEL_ID,
        'prompt':       'v3_simple',
    }) + '\n')

f_out.close()
print(f'\nDone. Output: {OUTPUT_FILE}')


In [ ]:
# Cell 8 — Quick scoring (Token F1)
import re
from collections import Counter

def norm(t):
    t = re.sub(r'[^\w\s]', ' ', str(t).lower())
    return re.sub(r'\s+', ' ', t).strip()

def token_f1(pred, gt):
    p, g = norm(pred).split(), norm(gt).split()
    if not p or not g: return 0.0
    pc, gc = Counter(p), Counter(g)
    common = sum((pc & gc).values())
    return 2 * common / (len(p) + len(g)) if common else 0.0

records = [json.loads(l) for l in open(OUTPUT_FILE)]
f1s, cls_c, cls_t, opn = [], [], [], []
for r in records:
    f1 = token_f1(r['prediction'], r['ground_truth'])
    f1s.append(f1)
    if r['is_closed']:
        p0 = norm(r['prediction']).split()
        g0 = norm(r['ground_truth']).split()
        cls_c.append(1 if (p0 and g0 and p0[0]==g0[0]) else 0)
        cls_t.append(f1)
    else:
        opn.append(f1)

def avg(l): return sum(l)/len(l)*100 if l else 0.0
print(f"HuatuoGPT-7B — SLAKE — v3_simple prompt")
print(f"  N total    : {len(records)}")
print(f"  Token F1   : {avg(f1s):.2f}%")
print(f"  Closed Acc : {avg(cls_c):.2f}%  (N={len(cls_c)})")
print(f"  Open F1    : {avg(opn):.2f}%   (N={len(opn)})")
print()
print(f"  Reference (v2 baseline, Section 11):")
print(f"  Token F1: 47.86%  |  Closed Acc: 72.36%  |  Open F1: ~32%")


## After running

Download `FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__slake_v3_corrected.jsonl`
from `/kaggle/working/outputs/` and place it in the project root.

Tell the agent — it will:
1. Run the LLM Judge on the new file (to get Judge Accuracy)
2. Compare v2 vs v3_corrected across all metrics
3. Update Section 19 / Section 11 in report.md with corrected numbers
4. Push to GitHub
